# Walmart Retail Sales Analysis

**Tools:** Python, pandas, NumPy, Matplotlib, SQL. This notebook analyzes the supplied Walmart.csv dataset.

In [ ]:
from pathlib import Path
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
ROOT = Path.cwd()
DATA = ROOT / 'data' / 'raw' / 'Walmart.csv'
df = pd.read_csv(DATA)
print('Shape:', df.shape)
df.head()

## 1. Data quality assessment

In [ ]:
df.info()
display(df.isna().sum().to_frame('missing_values'))
print('Exact duplicate rows:', df.duplicated().sum())
display(df.describe(include='all').T)

## 2. Data cleaning and feature engineering

The source file contains currency-formatted unit prices, date strings in day/month/year order, duplicate rows, and 31 missing price/quantity records. Missing revenue cannot be calculated, so those rows are retained in the all-rows cleaned file but excluded from revenue-based analysis.

In [ ]:
df.columns = [c.strip().lower() for c in df.columns]
df['unit_price'] = pd.to_numeric(df['unit_price'].astype(str).str.replace('$','',regex=False).str.replace(',','',regex=False), errors='coerce')
for c in ['quantity','rating','profit_margin']:
    df[c] = pd.to_numeric(df[c], errors='coerce')
df['date'] = pd.to_datetime(df['date'], dayfirst=True, errors='coerce')
duplicates_removed = int(df.duplicated().sum())
df = df.drop_duplicates().copy()
df['revenue'] = df['unit_price'] * df['quantity']
df['estimated_profit'] = df['revenue'] * df['profit_margin']
df['year'] = df['date'].dt.year
df['month'] = df['date'].dt.month
df['month_name'] = df['date'].dt.month_name()
df['day_name'] = df['date'].dt.day_name()
df['hour'] = pd.to_datetime(df['time'], format='%H:%M:%S', errors='coerce').dt.hour
sales = df.dropna(subset=['revenue']).copy()
print('Duplicates removed:', duplicates_removed, '| Revenue-valid records:', len(sales))

## 3. KPI summary

In [ ]:
kpis = pd.Series({'Revenue': sales.revenue.sum(), 'Estimated gross profit': sales.estimated_profit.sum(), 'Average transaction revenue': sales.revenue.mean(), 'Units sold': sales.quantity.sum(), 'Average customer rating': df.rating.mean()})
display(kpis.to_frame('value').round(2))

## 4. Category and branch performance

In [ ]:
category_sales = sales.groupby('category').revenue.sum().sort_values(ascending=False)
branch_sales = sales.groupby('branch').revenue.sum().sort_values(ascending=False)
display(category_sales.to_frame('revenue'))
display(branch_sales.head(10).to_frame('revenue'))
category_sales.sort_values().plot(kind='barh', figsize=(9,5), title='Revenue by Product Category'); plt.xlabel('Revenue'); plt.tight_layout(); plt.show()

## 5. Payment, time and yearly trends

In [ ]:
display(sales.payment_method.value_counts().to_frame('transactions'))
sales.groupby('year').revenue.sum().plot(kind='bar', title='Revenue by Year', figsize=(8,4)); plt.ylabel('Revenue'); plt.tight_layout(); plt.show()
sales.groupby('hour').revenue.sum().plot(kind='bar', title='Revenue by Hour', figsize=(9,4)); plt.ylabel('Revenue'); plt.tight_layout(); plt.show()

## 6. Export cleaned datasets

In [ ]:
processed = ROOT/'data'/'processed'; processed.mkdir(parents=True, exist_ok=True)
df.to_csv(processed/'walmart_cleaned_all_rows.csv', index=False)
sales.to_csv(processed/'walmart_revenue_analysis.csv', index=False)
print('Export complete.')

## 7. Business recommendations

- Protect availability and merchandising for the two leading revenue categories: Fashion accessories and Home and lifestyle.
- Compare lower-revenue categories by branch before changing inventory; revenue alone does not explain margin or demand.
- Review the 31 records with missing unit price/quantity at source rather than imputing financial values.
- Monitor branch-level performance with a consistent revenue and rating scorecard.
- Use payment-method shares to plan checkout experience, while avoiding assumptions about customer preferences beyond this dataset.